# Sesion 5: Execution modes

### 16.01.2026 14:30 CET

4.1. Execution modes such as: session with dedicated, priority, and batch mode


[https://quantum.cloud.ibm.com/docs/en/guides/execute-on-hardware](https://quantum.cloud.ibm.com/docs/en/guides/execute-on-hardware)

[https://quantum.cloud.ibm.com/docs/en/guides/execution-modes](https://quantum.cloud.ibm.com/docs/en/guides/execution-modes)

[https://quantum.cloud.ibm.com/docs/en/guides/choose-execution-mode](https://quantum.cloud.ibm.com/docs/en/guides/choose-execution-mode)

[https://quantum.cloud.ibm.com/docs/en/guides/run-jobs-batch](https://quantum.cloud.ibm.com/docs/en/guides/run-jobs-batch)

# How to configure API Key

If you are working in a trusted Python environment (such as on a personal laptop or workstation), you can use the save_account() method to save your credentials locally, then use them to initialize the service.

Find your API key. From the dashboard, create your API key, then copy it to a secure location so you can use it for authentication. Note that you can use the same API key to connect to any region.

[https://quantum.cloud.ibm.com/docs/en/guides/save-credentials](https://quantum.cloud.ibm.com/docs/en/guides/save-credentials)


# Introduction to Qiskit Runtime execution modes

there are three execution modes: job, session, and batch.

## Job ⚙️

The default mode for one-off executions. It runs a single estimator or sampler request without a context manager by packaging circuits and inputs into PUBs and submitting them as one execution task.

**Best for:** 
- Small tests, single circuit executions, or non-iterative workloads.
- Behavior: Each job is queued independently. There is no relationship between Job A and Job B.

**Use when:**
- You only need to run one job.
- You are testing or debugging circuits.




## Batch Mode 📦

<table style="width:100%; border-collapse: collapse; border: none;">
<tr>
<td style="width:66%; vertical-align:top; border: none;" >

**Best for:** Many independent jobs

- Submit multiple jobs together as a batch.
- All jobs enter the queue once and then run sequentially with minimal idle time.
- Classical preprocessing (such as compilation) is parallelized across jobs.
- Generally faster and more cost-effective than running jobs individually.
- Job execution order within a batch is not guaranteed.
- Other users’ jobs may run between yours.

**Use when:**
- Jobs are independent and known in advance.
- You want high throughput at lower cost.
- You are running parameter sweeps or comparisons.
</td>
<td style="width:34%; vertical-align:top; border: none;">
<img src="batch.png">
</td>
</tr>
</table>


You can open a runtime batch by using the context manager with Batch(...) or by initializing the Batch class. When you start a batch, you must specify a QPU by passing a backend object. The batch starts when its first job begins execution.

**Batch class**
```python
backend = service.least_busy(operational=True, simulator=False)
batch = Batch(backend=backend)
estimator = Estimator(mode=batch)
sampler = Sampler(mode=batch)
# Close the batch because no context manager was used.
batch.close()
```

**Context manager**
The context manager automatically opens and closes the batch.
```python
from qiskit_ibm_runtime import (
    Batch,
    SamplerV2 as Sampler,
    EstimatorV2 as Estimator,
)
 
backend = service.least_busy(operational=True, simulator=False)
with Batch(backend=backend):
    estimator = Estimator()
    sampler = Sampler()
```

In [1]:
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

# 1. Initialize Service
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# 2. Define Circuits
qc1 = QuantumCircuit(1); qc1.h(0); qc1.measure_all()
qc2 = QuantumCircuit(1); qc2.x(0); qc2.measure_all()

# 3. Transpile
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuits = [pm.run(qc1), pm.run(qc2)]

# 4. Execute with Batch and Print Details
with Batch(backend=backend) as batch:
    sampler = Sampler(mode=batch)
    
    # --- PRINT BATCH DETAILS ---
    print(f"--- Batch Initialization ---")
    print(f"Batch ID: {batch.session_id}")
    print(f"Current Status: {batch.status()}")
    
    # Run the circuits
    job = sampler.run(isa_circuits)
    print(f"Submitted Job ID: {job.job_id()}")
    
    # --- PRINT UPDATED DETAILS ---
    # details() returns a dictionary with metadata like usage_time and max_time
    details = batch.details()
    print(f"\n--- Detailed Metadata ---")
    print(f"Backend: {details.get('backend_name')}")
    print(f"State: {details.get('state')}")
    print(f"Max Time (s): {details.get('max_time')}")
    print(f"Usage Time (s): {details.get('usage_time')}")
    
    # Wait for results
    result = job.result()
    
    for i, pub_result in enumerate(result):
        print(f"Circuit {i+1} counts: {pub_result.data.meas.get_counts()}")

# Once outside the 'with' block, the batch is automatically closed.

qiskit_runtime_service.__init__:WARNING:2026-01-12 22:23:29,572: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: OneInstance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-01-12 22:23:30,334: Loading instance: OneInstance, plan: open
qiskit_runtime_service.backends:WARNING:2026-01-12 22:23:33,306: Using instance: OneInstance, plan: open


--- Batch Initialization ---
Batch ID: 4d1ab6ba-9010-459d-bad6-833bbab2be74
Current Status: Pending
Submitted Job ID: d5imckvea9qs7393e8eg

--- Detailed Metadata ---
Backend: ibm_torino
State: open
Max Time (s): 600
Usage Time (s): None
Circuit 1 counts: {'1': 1927, '0': 2169}
Circuit 2 counts: {'1': 3558, '0': 538}


## The Hierarchy of Choice

When you call a primitive like sampler.run(), Qiskit determines which execution mode to use based on where it finds the backend or session information. The order of priority is usually:

- Direct Arguments: Explicitly passing a backend or session to the Primitive constructor.
- Context Environment: The "active" context created by a with block.

❌ The Unsafe Way (Triggers Single Job Mode)

Even though the code is inside a Batch block, the sampler is "unaware" of it because it was given a backend explicitly.

```python
from qiskit_ibm_runtime import Batch, Sampler

with Batch(backend=backend) as batch:
    # This Sampler is NOT linked to the batch context!
    sampler = Sampler(backend=backend) 
    
    # This job will wait in the regular public queue
    job = sampler.run(circuit)
```

✅ The Safe Way (Option A: Implicit)

By not passing a backend to the Sampler, it automatically looks for an active context in the environment.

```python
with Batch(backend=backend) as batch:
    # No arguments needed; it detects the Batch context
    sampler = Sampler() 
    job = sampler.run(circuit)
```


✅ The Safe Way (Option B: Explicit Binding)

If you want to be 100% explicit for readability, you can pass the batch/session object itself into the mode parameter.

```python
with Batch(backend=backend) as batch:
    # Explicitly telling the sampler to use this specific batch
    sampler = Sampler(mode=batch) 
    job = sampler.run(circuit)
```

## Session

<table style="width:100%; border-collapse: collapse; border: none;">
<tr>
<td style="width:66%; vertical-align:top; border: none;" >

Designed for iterative algorithms where classical and quantum resources must communicate rapidly.

**Best for:** Iterative and interactive workflows

- Provides a **dedicated execution window** on a backend
- Multiple jobs can be run without re-entering the queue.
- Ideal for iterative or conditional algorithms (e.g., VQE, QAOA).
- More expensive than batch mode due to exclusive access.
- Offers little benefit for single jobs.
- **Not available for Open Plan users.**

**Use when:**
- Jobs depend on results from previous jobs.
- Low latency between iterations is critical.
</td>
<td style="width:34%; vertical-align:top; border: none;">
<img src="session.png">
</td>
</tr>
</table>

## Time to Live (TTL) ⏳

#Every Batch and Session is governed by two timers:

- **Maximum TTL** (max_time): The absolute wall-clock time a session/batch can stay open (starts when the first job starts).
- **Interactive TTL** (interactive_timeout): The "inactivity" timer (default 1 minute). If no new job is submitted within this window, the session/batch closes to free up the QPU for others.


---

## Summary 📌

| Feature | Single Job | Batch | Session |
| --- | --- | --- | --- |
| Primary Goal | Isolation | Throughput / Parallelism | Latency / Iteration |
| Dedicated Access | No | No | Yes |
| Ordering | FIFO Queue | Unordered/Parallel | Serial (Ordered) |
| First Job Wait | Normal Queue | Normal Queue | Normal Queue |
| Inspection Method | job.status() | batch.details() | session.details() |

---

## Example Use Cases 🧪

| Use Case | Recommended Mode |
|--------|------------------|
| Single experiment or test | Job |
| Parameter sweeps or comparisons | Batch |
| Variational algorithms (VQE, QAOA) | Session |
| Independent jobs known upfront | Batch |
| Iterative optimization loop | Session |
